# OrbitalDash in a Jupyter notebook

This notebook shows how to launch the OrbitalDash UI **from inside Jupyter**.

- Uses a `custom_eval(grid) -> (npoints, n_mo)` so it works **without PySCF**.
- If your installed Dash supports it, the app can run **inline** in the notebook.
- If inline mode is not supported by your Dash version, set `jupyter_mode='external'` (or `jupyter=False`) and open the printed URL in a browser.

In [ ]:
import numpy as np
from orbitaldash import launch_app

# H2 geometry in Angstrom
geometry = {"symbols": ["H", "H"], "coords": np.array([[0.0, 0.0, 0.0], [0.0, 0.0, 0.74]])}

# For a custom evaluator, mo_coeff is only used to determine n_mo (columns).
mo_coeff = np.eye(2)

sigma2 = 0.25  # width control

def custom_eval(grid: np.ndarray) -> np.ndarray:
    g1 = np.exp(-np.linalg.norm(grid - geometry["coords"][0], axis=1) ** 2 / sigma2)
    g2 = np.exp(-np.linalg.norm(grid - geometry["coords"][1], axis=1) ** 2 / sigma2)
    return np.stack([g1, g2], axis=1)

# Try inline first (if supported by your Dash version).
# If this doesn't embed, switch to jupyter_mode='external' or jupyter=False.
app = launch_app(
    geometry=geometry,
    basis="custom",
    mo_coeff=mo_coeff,
    custom_eval=custom_eval,
    jupyter=True,
    jupyter_mode="inline",
    host="127.0.0.1",
    port=8050,
    debug=False,
)

## Notes

- If you see a URL printed instead of an inline view, run again with `jupyter_mode='external'` and open the URL.
- If you get a port-in-use error, change `port=8050` to another value (e.g. `8051`).

In [ ]:
# Optional: PySCF example (requires `pip install -e .[full]` or installing pyscf separately)
try:
    from pyscf import gto, scf

    mol = gto.M(
        atom="N 0 0 0; N 0 0 1.0977",
        basis="cc-pvdz",
        unit="Angstrom",
        charge=0,
        spin=0,
    )
    mf = scf.RHF(mol).run()

    launch_app(
        mf,
        jupyter=True,
        jupyter_mode="inline",
        host="127.0.0.1",
        port=8052,
        debug=False,
    )
except Exception as e:
    print("PySCF example skipped (missing pyscf or failed run):", repr(e))